# Introduction


This Notebook introduces Gemma 4:2B (it).

We will test the multimodal and multilanguage capability of the model.


# Upgrade transformers

In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 70.9 MB/s eta 0:00:00
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


# Define a pipeline

In [2]:
from transformers import pipeline
pipe = pipeline("any-to-any", model="google/gemma-4-e2b-it")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

# Test with image

Let's use here the example from HuggingFace.
We will analyze an image.

[](https://huggingface.co/datasets/merve/vlm_test_images/resolve/main/thailand.jpg)

<img src="https://huggingface.co/datasets/merve/vlm_test_images/resolve/main/thailand.jpg"></img>

In [3]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": "https://huggingface.co/datasets/merve/vlm_test_images/resolve/main/thailand.jpg",
            },
            {"type": "text", "text": "Do you have travel advice going to here?"},
        ],
    }
]
output = pipe(messages, max_new_tokens=100, return_full_text=False)
output[0]["generated_text"]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Based on the image, you are looking at a magnificent **Buddhist temple or pagoda**, likely in **Southeast Asia**, given the architectural style (towers, intricate detailing, and the use of gold accents).\n\n**To give you specific and helpful travel advice, I need to know *where* this location is.**\n\nHowever, I can provide you with **general travel advice** based on the type of destination this photo suggests (a major Southeast Asian temple complex):\n\n---\n\n### General Travel Advice'

Let's beautify a bit the output.

In [4]:
from IPython.display import Markdown

display(Markdown(output[0]["generated_text"]))

Based on the image, you are looking at a magnificent **Buddhist temple or pagoda**, likely in **Southeast Asia**, given the architectural style (towers, intricate detailing, and the use of gold accents).

**To give you specific and helpful travel advice, I need to know *where* this location is.**

However, I can provide you with **general travel advice** based on the type of destination this photo suggests (a major Southeast Asian temple complex):

---

### General Travel Advice

As we limited the number of tokens to 100, the output message is not fully displayed.

# Test with video


We continue now also with the video from HuggingFace example.

Let's first display the video.

In [5]:
from IPython.display import Video

Video("https://huggingface.co/datasets/merve/vlm_test_images/resolve/main/rockets.mp4")

In [6]:
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "video",
                "video": "https://huggingface.co/datasets/merve/vlm_test_images/resolve/main/rockets.mp4",
            },
            {"type": "text", "text": "What is happening in this video?"},
        ],
    }
]

output = pipe(messages, load_audio_from_video=True)
output[0]["generated_text"]

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'role': 'user',
  'content': [{'type': 'video',
    'video': 'https://huggingface.co/datasets/merve/vlm_test_images/resolve/main/rockets.mp4'},
   {'type': 'text', 'text': 'What is happening in this video?'},
   {'type': 'audio'}]},
 {'role': 'assistant',
  'content': 'This video shows a crowd of people gathered on a tarmac, observing a large rocket, which appears to be a SpaceX Falcon 9 rocket, likely before or after a launch event. The sky is cloudy with soft, diffused light, suggesting either sunrise or sunset. There are also other aircraft and what look like exhibition booths in the background.'}]

Let's show the answer a bit beautified.

In [7]:
display(Markdown(output[0]["generated_text"][-1]["content"]))

This video shows a crowd of people gathered on a tarmac, observing a large rocket, which appears to be a SpaceX Falcon 9 rocket, likely before or after a launch event. The sky is cloudy with soft, diffused light, suggesting either sunrise or sunset. There are also other aircraft and what look like exhibition booths in the background.

Not sure how accurate is the answer, since the rocket shows Arianne logo, but the answer is close enough.

# More tests


Let's try also with another images.


## A cow on the beach

<img src="https://storage.googleapis.com/keras-cv/models/paligemma/cow_beach_1.png"></img>

In [8]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://storage.googleapis.com/keras-cv/models/paligemma/cow_beach_1.png"},
            {"type": "text", "text": "What you can see in this image?"}
        ]
    }
]

output = pipe(text=messages, max_new_tokens=300)
print(output[0]["generated_text"][-1]["content"])

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


This image features a **brown and white cow** standing on a **sandy beach**.

Here's a breakdown of what is visible:

* **Subject:** A bovine animal, likely a cow, with reddish-brown fur and a prominent white patch on its face. It appears to be looking towards the camera.
* **Setting (Foreground/Midground):** The cow is standing on light-colored sand. There is a wet or reflective area near its feet, suggesting the tide might be low or it has recently been near the water.
* **Setting (Background):** In the background, there is a body of **water** (likely the ocean or a large sea) with turquoise or blue-green colors, indicating clear, shallow water near the shore. Beyond the water, there are **landmasses** or **hills/mountains** visible on the horizon.
* **Sky:** The sky is bright **blue** with some scattered **white clouds**.
* **Atmosphere:** The lighting suggests it is a bright, sunny day, likely warm, given the beach setting.

In summary, it's a picturesque scene of a cow standing on

In [9]:
display(Markdown(output[0]["generated_text"][-1]["content"]))

This image features a **brown and white cow** standing on a **sandy beach**.

Here's a breakdown of what is visible:

* **Subject:** A bovine animal, likely a cow, with reddish-brown fur and a prominent white patch on its face. It appears to be looking towards the camera.
* **Setting (Foreground/Midground):** The cow is standing on light-colored sand. There is a wet or reflective area near its feet, suggesting the tide might be low or it has recently been near the water.
* **Setting (Background):** In the background, there is a body of **water** (likely the ocean or a large sea) with turquoise or blue-green colors, indicating clear, shallow water near the shore. Beyond the water, there are **landmasses** or **hills/mountains** visible on the horizon.
* **Sky:** The sky is bright **blue** with some scattered **white clouds**.
* **Atmosphere:** The lighting suggests it is a bright, sunny day, likely warm, given the beach setting.

In summary, it's a picturesque scene of a cow standing on a sunny beach overlooking the ocean.

# Small detail on a candy


<img src="https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"></img>

In [10]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is represented on the candy?"}
        ]
    }
]
output = pipe(text=messages, max_new_tokens=200)


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [11]:
display(Markdown(output[0]["generated_text"][-1]["content"]))

The candies appear to be shaped like **bees**.

Actually, the small drawing on the candies are more like turtles.

Let's check now both the counting abilities of this compact model as well as German language knowledge.

In [12]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "Welche Farben haben die Bombons?"}
        ]
    }
]
output = pipe(text=messages, max_new_tokens=200)


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [13]:
display(Markdown(output[0]["generated_text"][-1]["content"]))

Die Bombons auf dem Bild haben folgende Farben:

1. **Türkisgrün/Blaugrün** (zwei Stück)
2. **Orange** (ein Stück)
3. **Grün** (ein Stück)

The answer is perfect.

# Spanish culture and language


<img src="https://d1bv4heaa2n05k.cloudfront.net/user-images/1439905381602/shutterstock-78898486_destinationMain_1439905420657.jpeg"></img>

In [14]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://d1bv4heaa2n05k.cloudfront.net/user-images/1439905381602/shutterstock-78898486_destinationMain_1439905420657.jpeg"},
            {"type": "text", "text": "¿Qué ves en esta imagen?"}
        ]
    }
]
output = pipe(text=messages, max_new_tokens=200)


Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [15]:
display(Markdown(output[0]["generated_text"][-1]["content"]))

En esta imagen, se observa una escena que parece ser de una **exhibición o evento cultural**, posiblemente relacionado con la cultura del desierto o de alguna tradición equina/animal.

Aquí tienes una descripción detallada de lo que se ve:

1.  **Una mujer vestida con un traje llamativo:**
    *   Lleva un traje elaborado con muchos detalles dorados o de color amarillo/oro, que parece ser un traje de baile o de actuación tradicional.
    *   Viste pantalones ajustados de color oscuro (negro o morado oscuro).
    *   Viste medias o botas de color rosa brillante.
    *   Está envuelta en una gran tela o capa de color **púrpura o magenta intenso**, que se extiende dramáticamente a su alrededor.
    *   Su postura es dinámica, como si estuviera bailando o interactuando con el animal.

2.  **Un toro o ganado:**

The model is smart, but not that smart. It can interpret all the features and even small details in the image, but it is not recognizing a torrero scene.

# Japanese landmark

<img src="https://www.advantour.com/img/japan/tokyo/tokyo-tower.jpg"></img>

In [16]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://www.advantour.com/img/japan/tokyo/tokyo-tower.jpg"},
            {"type": "text", "text": "この画像には何が見えますか?"}
        ]
    }
]
output = pipe(text=messages, max_new_tokens=200)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [17]:
display(Markdown(output[0]["generated_text"][-1]["content"]))

この画像は、**東京の都市景観**を広角で捉えたものです。

画像から確認できる主な要素は以下の通りです。

1. **東京スカイツリー (Tokyo Skytree):** 画像の中央に非常に目立つ、高くそびえる赤い電波塔（タワー）が写っています。これが東京の象徴の一つです。
2. **都市のスカイライン:** スカイツリーの周囲には、高層ビルが密集した現代的な都市の風景が広がっています。様々な高さやデザインのビルが確認できます。
3. **広大な都市の範囲:** 背景には、遠くまで広がる都市の範囲と、地平線が見えています。
4. **自然の要素（緑）:** 都市の中に、豊かな緑の木々（特に前景や中景）が点在しており、都市と自然が共存している様子がうかがえます。
5. **空:** 空

The model was able to interpret correctly a landmark image from Tokyo, Tokyo Tower, with all details.

# French landmark

<img src="https://wmf.imgix.net/images/aa_fra_notre-dame_de_paris_0.jpg"></img>


In [18]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://wmf.imgix.net/images/aa_fra_notre-dame_de_paris_0.jpg"},
            {"type": "text", "text": "Que voyez-vous sur cette photo? Repondez succinte, svp."}
        ]
    }
]
output = pipe(text=messages, max_new_tokens=200)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [19]:
display(Markdown(output[0]["generated_text"][-1]["content"]))

Cette photo montre la **Cathédrale Notre-Dame de Paris** (ou une structure similaire de style gothique) sous un ciel bleu parsemé de nuages. On aperçoit également des bâtiments urbains et de la verdure sur les côtés.

The answer is quite good, recognizing the Notre Dame cathedral.

# Conclusions

We checked the multimodal and multilanguage features of Gemma 3:2B (it) model.

The model can interpret both image and video.

We could verify that the model is able to interpret correctly a variety of images (describe the content of an inedite scene, perceive small details in a picture, correctly identify a landmark) and is also capable to process the text (and output answer) in multiple languages. 

Being a small size model (2B), it miss some of the contextual information (e.g. did not recognized a corrida) but in general it is impressive by the capacity to deal with multi-modal and multi-language data.

We used English, German, French, Spanish, and Japanese.